# Feature Store Starter


<div align="center">
  <img src="./features_table.png" width="250">
</div>

## Overview
This notebook demonstrates a local Feature Store using **Feast** and real **NYC Taxi data**.

**Scenario:** We are building a model to predict tipping behavior. We need to store and retrieve features like `trip_distance` and `fare_amount` for specific drivers at specific points in time.

**Tech Stack:**
* **Feast**: Feature Store (Offline: Parquet, Online: SQLite)
* **Pandas**: Data Manipulation
* **Scikit-Learn**: Model Training

In [ ]:
# 1. Install Dependencies (if not already inst
!pip install feast pandas scikit-learn seaborn

## 1. Prepare the Data
Download the NYC Taxi dataset from GitHub. To make it compatible with a Feature Store, we need to:
1. **Shift Timestamps:** We shift the 2019 data to **Yesterday**. This ensures it is "fresh" (within the 1-year TTL) but definitely in the past so the store can ingest it safely.
2. **Add Entity ID:** We simulate driver IDs since the raw data doesn't have them.

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import os

# Load real NYC Taxi data from GitHub
print("⬇️ Downloading NYC Taxi Data...")
df = sns.load_dataset('taxis')

# Feature Engineering for Feast

# 1. Create Event Timestamp
raw_timestamps = pd.to_datetime(df['pickup'])
target_date = pd.Timestamp.now() - pd.Timedelta(days=1)
time_shift = target_date - raw_timestamps.max()
df['event_timestamp'] = raw_timestamps + time_shift

# 2. Generate synthetic Driver IDs (Entities)
df['driver_id'] = np.random.randint(1001, 1101, df.shape[0])

# 3. Select useful features
# Lets serve: distance, fare, and passenger count
feast_df = df[['driver_id', 'event_timestamp', 'distance', 'fare', 'passengers']]

# 4. Save to Parquet (The Offline Store)
os.makedirs("feature_repo/data", exist_ok=True)
parquet_path = "feature_repo/data/taxi_stats.parquet"
feast_df.to_parquet(parquet_path)

print(f"✅ Data prepared at {parquet_path}")
print(feast_df[['driver_id', 'event_timestamp']].tail())

## 2. Define the Feature Repository
Feast uses a Python file to define your features. We will programmatically create `feature_repo/example.py` to point to our new Parquet file.

In [ ]:
feature_def = """
from datetime import timedelta
from feast import Entity, Field, FeatureView, FileSource, ValueType
from feast.types import Float32, Int64

# 1. Define the Entity (The Driver)
driver = Entity(name="driver", join_keys=["driver_id"])

# 2. Define the Data Source (Our Parquet File)
taxi_stats_source = FileSource(
    name="taxi_stats_source",
    path="data/taxi_stats.parquet",
    timestamp_field="event_timestamp",
)

# 3. Define the Feature View
# This groups the features together and defines their types
taxi_stats_fv = FeatureView(
    name="taxi_driver_stats",
    entities=[driver],
    ttl=timedelta(days=365),
    schema=[
        Field(name="distance", dtype=Float32),
        Field(name="fare", dtype=Float32),
        Field(name="passengers", dtype=Int64),
    ],
    online=True,
    source=taxi_stats_source,
)
"""

# Write the definition file
with open("feature_repo/example.py", "w") as f:
    f.write(feature_def)

# Create the feature_store.yaml configuration
yaml_config = """
project: taxi_feature_store
registry: data/registry.db
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
"""

with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(yaml_config)

print("✅ Feature Repository configured.")

## 3. Apply and Materialize
We register the features and then **Materialize** (Load) the data.

**Note:** We use a forceful materialization window (`start` to `end`) to guarantee that our newly created "Yesterday" data is loaded, regardless of any previous runs.

In [ ]:
# Switch to the repo directory
%cd feature_repo

# 1. Apply: Register the feature definitions
!feast apply

# 2. Materialize: Load data from Parquet (Offline) to SQLite (Online)
from datetime import datetime, timedelta

# Define the window: Look back 5 days and look forward 1 day
start_date = (datetime.now() - timedelta(days=5)).isoformat()
end_date = (datetime.now() + timedelta(days=1)).isoformat()

print(f"🚀 Materializing data from {start_date} to {end_date}...")
!feast materialize {start_date} {end_date}

## 4. Retrieve Features
Let's can now fetch features in two ways:
1. **Historical Retrieval**: For training a model (Time-travel join).
2. **Online Retrieval**: For real-time inference (Latest values).

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=".")

# --- 1. Historical Retrieval (Training Data) ---
entity_df = pd.DataFrame.from_dict({
    "driver_id": [1001, 1002, 1001], 
    "event_timestamp": [
        pd.Timestamp.now() - pd.Timedelta(hours=20),
        pd.Timestamp.now() - pd.Timedelta(hours=20),
        pd.Timestamp.now() - pd.Timedelta(hours=20)
    ]
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "taxi_driver_stats:distance",
        "taxi_driver_stats:fare"
    ]
).to_df()

print("\n--- Historical Training Data (Past) ---")
print(training_df.head())

# --- 2. Online Retrieval (Inference) ---
online_features = store.get_online_features(
    features=[
        "taxi_driver_stats:distance",
        "taxi_driver_stats:fare",
        "taxi_driver_stats:passengers"
    ],
    entity_rows=[
        {"driver_id": 1001},
        {"driver_id": 1002}
    ]
).to_dict()

print("\n--- Online Inference Features (Latest Valid Values) ---")
print(online_features)

## 5. Conclusion & Next Steps

Congratulations! You have successfully built a local Feature Store pipeline using **Feast** and real-world data.

### 📚 Resources
* **Infrastructure**: This template is optimized for [Saturn Cloud](https://saturncloud.io/), which allows you to scale this workflow from a single notebook to a production-grade cluster.
* **Dataset**: The NYC Taxi data used here is hosted publicly by the [Seaborn Data Repository](https://github.com/mwaskom/seaborn-data).
* **Feast Docs**: Dive deeper into Feature Views and Entity management at [feast.dev](https://feast.dev/).